# Lab 4.4 · Review a peer's product

Day 4 · Session 4 (Design review) · 25 min. Team B presents its governed analytics product for billing and
collections to the review panel; you sit on the panel. You run the review checks on their package, add what the
checks cannot see, rank every finding by risk, and log the panel's decisions. This is a rehearsal of the
capstone's design review, from the other chair.

**AI-tutor.** An assistant may help you read Team B's files; the findings, their ranking and the decisions are yours.

**Reference state.** Team B's package in `data/review/`: `README.md`, `model.yaml`, `metrics.yml`, `policy.yaml`,
`evaluation.csv` and `routing.yaml` (synthetic). It runs on the same Qamar marts.

**Tasks**
- [ ] 1 · Run the review checks (6')
- [ ] 2 · What the checks cannot see (6')
- [ ] 3 · Rank the findings by risk (7')
- [ ] 4 · Log the panel's decisions (6')
- [ ] ✱ Stretch · The same checklist on your own capstone plan

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/ds-408.git"
    if not Path("/content/ds-408").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/ds-408"], check=True)
    os.chdir("/content/ds-408/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not Path("qamar.py").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import math
import re
import pandas as pd
import qamar as Q
import qsem as S

PKG = Path("data/review")
print((PKG / "README.md").read_text(encoding="utf-8"))
con = Q.db()
POLICY = S.load_yaml(PKG / "policy.yaml")
DOC = S.with_metrics(S.load_yaml(PKG / "model.yaml"), S.load_yaml(PKG / "metrics.yml"))
peer = S.Model(DOC, con, policy=POLICY, **POLICY.get("settings", {}))
EVAL = pd.read_csv(PKG / "evaluation.csv")
ROUTING = S.load_yaml(PKG / "routing.yaml")
DUCKDB_ONLY = ["date_diff", "split_part", "strftime", "list_", "regexp_extract", "epoch_ms"]


def finding(lens: str, title: str, evidence: str) -> dict:
    return {"lens": lens, "finding": title, "evidence": evidence}


def model_checks(model) -> list[dict]:
    """The supplied checks of the model, the access rules and portability (lenses: correctness, governance, portability)."""
    out = []
    tests = S.run_tests(model)
    out.append(finding("correctness", "reconciliation tests", f"{int(tests.passed.sum())} of {len(tests)} pass"))
    total = con.execute("SELECT SUM(data_gb) FROM fct_usage_monthly").fetchone()[0]
    by_plan = model.query(["data_gb"], by=["tariff.plan_name"]).data_gb.sum()
    if abs(by_plan - total) > 0.01:
        out.append(finding("correctness", "data by plan does not add up to the total", f"{by_plan:,.0f} GB against {total:,.0f} GB"))
    mgr = model.as_role("regional_manager_eas")
    regions = sorted(mgr.query(["complaints"], by=["complaint.region"])["complaint.region"].dropna().unique())
    if regions != ["EAS"]:
        out.append(finding("governance", "the Eastern manager sees complaints of other regions", f"regions returned: {regions}"))
    names = model.as_role("executive").query(["billed_amount"], by=["customer.contact_name"]).head(3)["customer.contact_name"].tolist()
    if any(n != "***" for n in names):
        out.append(finding("governance", "a field derived from a Restricted column is readable", f"contact_name: {names}"))
    for d in model.doc["datasets"]:
        for f in d.get("fields", []):
            hits = [fn for fn in DUCKDB_ONLY if re.search(rf"\b{fn}", f["sql"])]
            if hits:
                out.append(finding("portability", f"DuckDB-only SQL in {d['name']}.{f['name']}", f"{hits}: {f['sql']}"))
    return out


checks = model_checks(peer)
print(pd.DataFrame(checks).to_string(index=False, max_colwidth=80))

## 1 · Run the review checks (6')

The model, access and portability checks above are supplied. The evaluation and the routing need checks too.

✏️ **Exercise** · Write `evaluation_checks(ev, routing)` returning findings (lens `evaluation` or `routing`) for:
fewer than 40 questions; one language only; fewer than three runs; no must-refuse question; and any type routed
to the assistant whose Wilson 95% upper bound on wrong figures is above the tolerance.

In [ ]:
def wilson_upper(k, n, z=1.96):
    p = k / n
    return (p + z * z / (2 * n) + z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / (1 + z * z / n)


def evaluation_checks(ev: pd.DataFrame, routing: dict) -> list[dict]:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    checks2 = evaluation_checks(EVAL, ROUTING)
except NotImplementedError:
    checks2 = Q.reference("d4_4_eval_checks")
print(pd.DataFrame(checks2).to_string(index=False))
Q.check(len(checks2) == 7 and {c["lens"] for c in checks2} == {"evaluation", "routing"}, "four evaluation findings and three routing findings")
found = [c for c in checks + checks2 if not c["finding"].startswith("reconciliation tests")]

🔍 Every reconciliation test of Team B passes. Does that make their metrics correct? Read their `collection_rate`
in `data/review/metrics.yml` and its test before step 2.

## 2 · What the checks cannot see (6')

Checks find what someone thought to check. A panel reads for meaning: is each certified metric the definition the
business agreed (Day 1)? Does each claim in the README come with the way to reproduce it?

✏️ **Exercise** · Return the findings the checks missed, each with `lens` (correctness, governance, evaluation or
reproducibility), `finding` and `evidence` quoted from Team B's files. Find at least two.

In [ ]:
def beyond_checks() -> list[dict]:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    extra = beyond_checks()
except NotImplementedError:
    extra = Q.reference("d4_4_beyond")
print(pd.DataFrame(extra).to_string(index=False, max_colwidth=90))
Q.check(len(extra) >= 2 and any("collection_rate" in f["finding"] for f in extra), "the panel finds the collection rate the checks passed")
findings = found + extra

## 3 · Rank the findings by risk (7')

| Score | Impact | Likelihood |
|---|---|---|
| 3 | a wrong figure or a leak reaches managers without anyone noticing | on most questions of a common type |
| 2 | a visible failure, or a wrong figure someone would notice | on some questions or one role |
| 1 | documentation, cost or a future move of tools | rarely, or only after a change |

✏️ **Exercise** · Give each finding an `impact` and a `likelihood` (write your scores in `SCORES`, keyed by the
finding's text), compute `risk` as their product, and return the findings sorted by risk, highest first.

In [ ]:
def rank(findings: list[dict]) -> pd.DataFrame:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    ranked = rank(findings)
except NotImplementedError:
    ranked = pd.DataFrame(Q.reference("d4_4_ranked"))
print(ranked[["risk", "impact", "likelihood", "lens", "finding"]].to_string(index=False))
top2 = set(ranked.finding.head(2))
Q.check(top2 == {"collection_rate is finance's cash-over-billed ratio, marked certified", "the Eastern manager sees complaints of other regions"},
        "the silent wrong figure and the silent leak rank first")

🔍 The portability findings rank last. When would they rank first? (Think of Team B's plans for next year.)

## 4 · Log the panel's decisions (6')

✏️ **Exercise** · Add `decision` (`fix before release` for risk 6 or more, `fix with an owner and a date` for 3 to 5,
`accept and record` below 3), `owner` (Team B's role for the lens: correctness the metrics owner, governance the
data protection officer, evaluation and routing the product owner, portability and reproducibility the platform
lead) and `due`. Save `decision_log.csv` and return it.

In [ ]:
OWNERS = {"correctness": "metrics owner", "governance": "data protection officer", "evaluation": "product owner",
          "routing": "product owner", "portability": "platform lead", "reproducibility": "platform lead"}


def decision_log(ranked: pd.DataFrame) -> pd.DataFrame:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    log = decision_log(ranked)
except NotImplementedError:
    log = pd.DataFrame(Q.reference("d4_4_log"))
print(log[["risk", "finding", "decision", "owner", "due"]].to_string(index=False))
log.to_csv(Q.lab_dir("D4_4") / "decision_log.csv", index=False, lineterminator="\n")
Q.check(not ((log.risk >= 6) & (log.decision != "fix before release")).any() and log.owner.notna().all(),
        "every high-risk finding is fixed before release and every finding has an owner")
print("panel verdict:", "not ready for release" if (log.decision == "fix before release").any() else "ready")

## ✱ Stretch · The same checklist on your own capstone plan

Run `model_checks` and `evaluation_checks` on the supplied Qamar model (`data/semantic/qamar_model.yaml` with the
governed policy) and on `data/assistant/evaluation.csv`. Which findings remain, and which belong in your capstone
README as known limits?

In [ ]:
def stretch():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")


try:
    stretch()
except NotImplementedError:
    print("✱ Stretch not attempted")

## Checkpoint

1. All of Team B's tests pass. Why did the panel still rank a metric finding first?

<details><summary>Answer</summary>A test that reconciles a metric to its own formula cannot find a wrong
definition; only a reader who knows the agreed definition can. Passing tests are evidence of consistency, not of
correctness.</details>

2. Why log an owner and a date for each finding, not just the finding?

<details><summary>Answer</summary>A finding without an owner is not fixed; the log is what the next review
checks against.</details>

**Carry forward.** `artifacts/D4_4/decision_log.csv`. In the capstone you sit in Team B's chair.